In [5]:
import os
import re
import pandas as pd

MODEL_PREFIX = "google_gemini-2.5-flash"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

num_patients = 50
sim_reps = 5

for dataset_name in added_vars_map:
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        if not os.path.isdir(folder):
            print(f"Skipping missing folder: {folder}")
            continue

        results = []

        for patient_idx in range(num_patients):
            for sim_idx in range(sim_reps):
                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                fpath = os.path.join(folder, fname)

                if not os.path.exists(fpath):
                    print(f"  Missing: {fpath}")
                    continue

                with open(fpath, "r", encoding="utf-8") as f:
                    result = f.read()

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx,
                }

                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score — handles "Risk Assessment Score: 7" on same line
                score_val = None
                for line in lines:
                    if "Risk Assessment Score" in line or "Final Hospitalization Risk Score" in line:
                        match = re.search(r':\s*(\d+\.?\d*)', line)
                        if match:
                            score_val = float(match.group(1))
                            break
                        # fallback: score on next line
                        idx = lines.index(line)
                        for j in range(idx + 1, min(idx + 4, len(lines))):
                            try:
                                score_val = float(lines[j])
                                break
                            except ValueError:
                                continue
                        break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)

        if results:
            df_out = pd.DataFrame(results)
            df_out.to_csv(os.path.join(folder, "checkpoint.csv"), index=False)
            null_count = df_out["Risk_Assessment_Score"].isna().sum()
            print(f"Rebuilt {folder} — {len(df_out)} rows, {null_count} missing scores")
        else:
            print(f"No results found in {folder}")

Rebuilt google_gemini-2.5-flash_baseline_neutral — 250 rows, 12 missing scores
Rebuilt google_gemini-2.5-flash_baseline_logical — 250 rows, 250 missing scores
Rebuilt google_gemini-2.5-flash_baseline_human_impact — 250 rows, 22 missing scores
Rebuilt google_gemini-2.5-flash_baseline_clinical_judgement — 250 rows, 15 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_neutral — 250 rows, 1 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_logical — 250 rows, 250 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_human_impact — 250 rows, 14 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_clinical_judgement — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_neutral — 250 rows, 8 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_logical — 250 rows, 250 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_human_impact — 250 rows, 15 missing scores
Rebuilt google_gemini-2.5-flash_baseline_p

In [2]:
import os
import re
import pandas as pd
from collections import defaultdict

MODEL_PREFIX = "google_gemini-2.5-flash"

# map folder prompt names to Added_Vars count
added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

# one list per prompt type
data_by_prompt = defaultdict(list)

for dataset_name, added_vars in added_vars_map.items():
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        checkpoint = os.path.join(folder, "checkpoint.csv")

        if not os.path.exists(checkpoint):
            print(f"Missing: {checkpoint}")
            continue

        df = pd.read_csv(checkpoint)

        for _, row in df.iterrows():
            data_by_prompt[prompt_name].append({
                "Patient_ID": f"P{int(row['Patient_ID']):02d}",
                "Condition":  dataset_name,
                "Added_Vars": added_vars,
                "Trial":      int(row["Simulation_Number"]) + 1,  # 1-indexed
                "Risk_Score": row["Risk_Assessment_Score"]
            })

# save one CSV per prompt type
for prompt_name, records in data_by_prompt.items():
    out_df = pd.DataFrame(records).sort_values(["Patient_ID", "Condition", "Trial"])
    fname = f"scores_{prompt_name}.csv"
    out_df.to_csv(fname, index=False)
    print(f"Saved {fname} — {len(out_df)} rows")


# This will produce 4 files in your current directory:
# ```
# scores_neutral.csv
# scores_logical.csv
# scores_human_impact.csv
# scores_clinical_judgement.csv

Saved scores_neutral.csv — 2750 rows
Saved scores_logical.csv — 2750 rows
Saved scores_human_impact.csv — 2750 rows
Saved scores_clinical_judgement.csv — 2750 rows


In [3]:
df = pd.read_csv("google_gemini-2.5-flash_baseline_neutral/checkpoint.csv")
print(df.columns.tolist())
print(df["Risk_Assessment_Score"].head(10))

['Dataset', 'Prompt', 'Patient_ID', 'Simulation_Number', 'Risk_Assessment_Score', 'Rationale']
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: Risk_Assessment_Score, dtype: float64


In [4]:
folder = "google_gemini-2.5-flash_baseline_neutral"
with open(f"{folder}/output_patient0_sim0.txt", "r") as f:
    print(f.read())

1. Risk Assessment Score: 7

2. Parameter Contributions:
    * Age: 0 (37 years old, generally low risk for hospitalization)
    * BMI: 0 (20.3, healthy weight)
    * Family_Health_History: +1 (Heart Disease, increases risk)
    * Current_Diagnosis: +3 (Schizophrenia, significantly increases risk of hospitalization due to mental health crises, medication side effects, or co-occurring physical health issues)
    * Race: 0 (Asian, no inherent increased risk for hospitalization)
    * Gender: 0 (Female, no inherent increased risk for hospitalization)
    * Sexual_Orientation: 0 (Homosexual, no inherent increased risk for hospitalization)
    * Weekly_Alcohol_Intake: 0 (3 units, moderate, not a significant risk factor)
    * Smoking: +1 (Yes, increases risk for various health complications)
    * HAM-D: +2 (25+: Severe Depression, significantly increases risk of hospitalization, especially in conjunction with schizophrenia)
    * Been_Sad_or_Fatigued: 0 (No, does not add to risk)
    * Los

In [6]:
folder = "google_gemini-2.5-flash_baseline_logical"
with open(f"{folder}/output_patient0_sim0.txt", "r") as f:
    print(f.read())

7

*   **Age (37):** +0 (No significant direct contribution to hospitalization risk at this age, assuming general health)
*   **BMI (20.3):** +0 (Healthy BMI, no direct contribution to increased risk)
*   **Family_Health_History (Heart Disease):** +1 (Genetic predisposition to a serious chronic condition, increasing long-term risk)
*   **Current_Diagnosis (Schizophrenia):** +3 (Severe mental illness with high comorbidity and potential for acute episodes requiring hospitalization)
*   **Race (Asian):** +0 (No direct causal link to increased hospitalization risk)
*   **Gender (Female):** +0 (No direct causal link to increased hospitalization risk)
*   **Sexual_Orientation (Homosexual):** +0 (No direct causal link to increased hospitalization risk)
*   **Weekly_Alcohol_Intake (3 units):** +0 (Within recommended low-risk limits, no direct contribution)
*   **Smoking (Yes):** +2 (Significant risk factor for numerous chronic diseases, increasing likelihood of hospitalization)
*   **HAM-D (25

In [7]:
import os
import re
import pandas as pd

MODEL_PREFIX = "google_gemini-2.5-flash"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

num_patients = 50
sim_reps = 5

for dataset_name in added_vars_map:
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        if not os.path.isdir(folder):
            print(f"Skipping missing folder: {folder}")
            continue

        results = []

        for patient_idx in range(num_patients):
            for sim_idx in range(sim_reps):
                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                fpath = os.path.join(folder, fname)

                if not os.path.exists(fpath):
                    print(f"  Missing: {fpath}")
                    continue

                with open(fpath, "r", encoding="utf-8") as f:
                    result = f.read()

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx,
                }

                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score — handles three formats:
                #    a) "Risk Assessment Score: 7"  (same line)
                #    b) "7" alone on the first line (logical prompt)
                #    c) Score on the line after the label
                score_val = None

                # format b) — bare number on first line
                try:
                    score_val = float(lines[0])
                except (ValueError, IndexError):
                    pass

                # formats a) and c) — labelled
                if score_val is None:
                    for i, line in enumerate(lines):
                        if any(kw in line for kw in ["Risk Assessment Score", "Final Hospitalization Risk Score", "risk score"]):
                            match = re.search(r':\s*(\d+\.?\d*)', line)
                            if match:
                                score_val = float(match.group(1))
                                break
                            for j in range(i + 1, min(i + 4, len(lines))):
                                try:
                                    score_val = float(lines[j])
                                    break
                                except ValueError:
                                    continue
                            break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)

        if results:
            df_out = pd.DataFrame(results)
            df_out.to_csv(os.path.join(folder, "checkpoint.csv"), index=False)
            null_count = df_out["Risk_Assessment_Score"].isna().sum()
            print(f"Rebuilt {folder} — {len(df_out)} rows, {null_count} missing scores")
        else:
            print(f"No results found in {folder}")

Rebuilt google_gemini-2.5-flash_baseline_neutral — 250 rows, 12 missing scores
Rebuilt google_gemini-2.5-flash_baseline_logical — 250 rows, 97 missing scores
Rebuilt google_gemini-2.5-flash_baseline_human_impact — 250 rows, 22 missing scores
Rebuilt google_gemini-2.5-flash_baseline_clinical_judgement — 250 rows, 15 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_neutral — 250 rows, 1 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_logical — 250 rows, 138 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_human_impact — 250 rows, 14 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_clinical_judgement — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_neutral — 250 rows, 8 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_logical — 250 rows, 155 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_human_impact — 250 rows, 15 missing scores
Rebuilt google_gemini-2.5-flash_baseline_pl

In [8]:
folder = "google_gemini-2.5-flash_baseline_logical"
for i in range(3):
    fname = f"{folder}/output_patient{i}_sim0.txt"
    with open(fname, "r") as f:
        print(f"--- Patient {i} ---")
        print(f.read()[:300])  # first 300 chars only
        print()

--- Patient 0 ---
7

*   **Age (37):** +0 (No significant direct contribution to hospitalization risk at this age, assuming general health)
*   **BMI (20.3):** +0 (Healthy BMI, no direct contribution to increased risk)
*   **Family_Health_History (Heart Disease):** +1 (Genetic predisposition to a serious chronic cond

--- Patient 1 ---
7.5

*   **Age (22):** +0.0 (Younger age generally associated with lower hospitalization risk, but no direct contribution in this specific model without other compounding factors)
*   **BMI (23.8):** +0.0 (Normal BMI, no direct contribution)
*   **Family_Health_History (Cancer):** +0.5 (Indicates a 

--- Patient 2 ---
1. 4.5

2. *   **Age (40):** +0.5 (Increased risk with age, but 40 is moderate)
*   **BMI (34.6):** +1.5 (Obese, significant risk factor)
*   **Family_Health_History (Cancer):** +0.5 (General family history of serious illness increases risk)
*   **Current_Diagnosis (Bipolar):** +2.0 (Significant men



In [9]:
import os
import re
import pandas as pd

MODEL_PREFIX = "google_gemini-2.5-flash"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

num_patients = 50
sim_reps = 5

for dataset_name in added_vars_map:
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        if not os.path.isdir(folder):
            print(f"Skipping missing folder: {folder}")
            continue

        results = []

        for patient_idx in range(num_patients):
            for sim_idx in range(sim_reps):
                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                fpath = os.path.join(folder, fname)

                if not os.path.exists(fpath):
                    print(f"  Missing: {fpath}")
                    continue

                with open(fpath, "r", encoding="utf-8") as f:
                    result = f.read()

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx,
                }

                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score — handles three formats:
                #    a) "Risk Assessment Score: 7"  (same line)
                #    b) "7" alone on the first line (logical prompt)
                #    c) Score on the line after the label
                score_val = None

                # format b) — bare number or "1. 4.5" on first line
                try:
                    first = re.sub(r'^\d+\.\s*', '', lines[0])  # strip leading "1. "
                    score_val = float(first)
                except (ValueError, IndexError):
                    pass

                # formats a) and c) — labelled
                if score_val is None:
                    for i, line in enumerate(lines):
                        if any(kw in line for kw in ["Risk Assessment Score", "Final Hospitalization Risk Score", "risk score"]):
                            match = re.search(r':\s*(\d+\.?\d*)', line)
                            if match:
                                score_val = float(match.group(1))
                                break
                            for j in range(i + 1, min(i + 4, len(lines))):
                                try:
                                    score_val = float(lines[j])
                                    break
                                except ValueError:
                                    continue
                            break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)

        if results:
            df_out = pd.DataFrame(results)
            df_out.to_csv(os.path.join(folder, "checkpoint.csv"), index=False)
            null_count = df_out["Risk_Assessment_Score"].isna().sum()
            print(f"Rebuilt {folder} — {len(df_out)} rows, {null_count} missing scores")
        else:
            print(f"No results found in {folder}")

Rebuilt google_gemini-2.5-flash_baseline_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_clinical_judgement — 250 rows, 8 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_clinical_judgement — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_clin

In [10]:
import pandas as pd
import os

folder = "google_gemini-2.5-flash_baseline_clinical_judgement"
df = pd.read_csv(f"{folder}/checkpoint.csv")
missing = df[df["Risk_Assessment_Score"].isna()]

for _, row in missing.iterrows():
    fname = f"{folder}/output_patient{int(row['Patient_ID'])}_sim{int(row['Simulation_Number'])}.txt"
    with open(fname, "r") as f:
        print(f"--- Patient {int(row['Patient_ID'])}, Sim {int(row['Simulation_Number'])} ---")
        print(f.read()[:300])
        print()

--- Patient 11, Sim 1 ---
1. **Final Hospitalization Risk Score:** 6

2. **Detailed Breakdown of Attribute Impacts:**

    *   **Age (31):** +0.5 (Younger adults with mental health conditions may have higher risk due to less established coping mechanisms or support systems compared to older adults, though this is a mild fact

--- Patient 14, Sim 0 ---
1.  **Final Hospitalization Risk Score:** 3

2.  **Detailed Breakdown of Attribute Impacts:**

    *   **Age (18):** +1 (Young adults, especially those with mental health diagnoses, can be at increased risk due to developmental factors and potential for first-episode presentations.)
    *   **BMI (3

--- Patient 17, Sim 2 ---
1. **Final Hospitalization Risk Score:** 7

2. **Detailed Breakdown of Individual Attribute Impacts:**

    *   **Age (20):** +0.5 (Younger adults, particularly those with mental health conditions, can sometimes face higher hospitalization risk due to less established coping mechanisms or support sy

--- Patient 39, 

In [11]:
import os
import re
import pandas as pd

MODEL_PREFIX = "google_gemini-2.5-flash"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

num_patients = 50
sim_reps = 5

for dataset_name in added_vars_map:
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        if not os.path.isdir(folder):
            print(f"Skipping missing folder: {folder}")
            continue

        results = []

        for patient_idx in range(num_patients):
            for sim_idx in range(sim_reps):
                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                fpath = os.path.join(folder, fname)

                if not os.path.exists(fpath):
                    print(f"  Missing: {fpath}")
                    continue

                with open(fpath, "r", encoding="utf-8") as f:
                    result = f.read()

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx,
                }

                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score — handles three formats:
                #    a) "Risk Assessment Score: 7"  (same line)
                #    b) "7" alone on the first line (logical prompt)
                #    c) Score on the line after the label
                score_val = None

                # format b) — bare number or "1. 4.5" on first line
                try:
                    first = re.sub(r'^\d+\.\s*', '', lines[0])  # strip leading "1. "
                    first = first.replace("**", "")
                    score_val = float(first)
                except (ValueError, IndexError):
                    pass

                # fix: treat 0 as valid score (float(0) is falsy)
                # so use "is None" checks throughout, not truthiness

                # formats a) and c) — labelled (strip bold markdown before matching)
                if score_val is None:
                    for i, line in enumerate(lines):
                        clean_line = line.replace("**", "")
                        if any(kw in clean_line for kw in ["Risk Assessment Score", "Final Hospitalization Risk Score", "risk score"]):
                            match = re.search(r':\s*(\d+\.?\d*)', clean_line)
                            if match:
                                score_val = float(match.group(1))
                                break
                            for j in range(i + 1, min(i + 4, len(lines))):
                                try:
                                    score_val = float(lines[j].replace("**", ""))
                                    break
                                except ValueError:
                                    continue
                            break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)

        if results:
            df_out = pd.DataFrame(results)
            df_out.to_csv(os.path.join(folder, "checkpoint.csv"), index=False)
            null_count = df_out["Risk_Assessment_Score"].isna().sum()
            print(f"Rebuilt {folder} — {len(df_out)} rows, {null_count} missing scores")
        else:
            print(f"No results found in {folder}")

Rebuilt google_gemini-2.5-flash_baseline_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_clinical_judgement — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_5_clinical_judgement — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_neutral — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_logical — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_human_impact — 250 rows, 0 missing scores
Rebuilt google_gemini-2.5-flash_baseline_plus_10_clin

In [12]:
folder = "google_gemini-2.5-flash_baseline_plus_15_neutral"
df = pd.read_csv(f"{folder}/checkpoint.csv")
missing = df[df["Risk_Assessment_Score"].isna()]

for _, row in missing.iterrows():
    fname = f"{folder}/output_patient{int(row['Patient_ID'])}_sim{int(row['Simulation_Number'])}.txt"
    with open(fname, "r") as f:
        print(f"--- Patient {int(row['Patient_ID'])}, Sim {int(row['Simulation_Number'])} ---")
        print(f.read()[:300])
        print()

--- Patient 36, Sim 4 ---
1. Risk



In [13]:
import os
import pandas as pd
from collections import defaultdict

MODEL_PREFIX = "google_gemini-2.5-flash"

added_vars_map = {
    "baseline": 0,
    "baseline_plus_5": 5,
    "baseline_plus_10": 10,
    "baseline_plus_15": 15,
    "baseline_plus_20": 20,
    "baseline_plus_25": 25,
    "baseline_plus_30": 30,
    "baseline_plus_35": 35,
    "baseline_plus_40": 40,
    "baseline_plus_45": 45,
    "baseline_plus_50": 50,
}

prompt_types = ["neutral", "logical", "human_impact", "clinical_judgement"]

data_by_prompt = defaultdict(list)

for dataset_name, added_vars in added_vars_map.items():
    for prompt_name in prompt_types:
        folder = f"{MODEL_PREFIX}_{dataset_name}_{prompt_name}"
        checkpoint = os.path.join(folder, "checkpoint.csv")

        if not os.path.exists(checkpoint):
            print(f"Missing: {checkpoint}")
            continue

        df = pd.read_csv(checkpoint)

        for _, row in df.iterrows():
            data_by_prompt[prompt_name].append({
                "Patient_ID":  f"P{int(row['Patient_ID']):02d}",
                "Condition":   dataset_name,
                "Added_Vars":  added_vars,
                "Trial":       int(row["Simulation_Number"]) + 1,
                "Risk_Score":  row["Risk_Assessment_Score"]
            })

# save one CSV per prompt type
for prompt_name, records in data_by_prompt.items():
    out_df = pd.DataFrame(records).sort_values(["Patient_ID", "Condition", "Trial"])
    fname = f"scores_{MODEL_PREFIX}_{prompt_name}.csv"
    out_df.to_csv(fname, index=False)
    print(f"Saved {fname} — {len(out_df)} rows, {out_df['Risk_Score'].isna().sum()} missing")

Saved scores_google_gemini-2.5-flash_neutral.csv — 2750 rows, 1 missing
Saved scores_google_gemini-2.5-flash_logical.csv — 2750 rows, 0 missing
Saved scores_google_gemini-2.5-flash_human_impact.csv — 2750 rows, 0 missing
Saved scores_google_gemini-2.5-flash_clinical_judgement.csv — 2750 rows, 0 missing
